# PyDESeq2 script for bulk RNA sequencing analysis.

In [ ]:
import os
# Project data root. Override with the ACC_DATA_ROOT environment variable.
ACC_DATA_ROOT = os.environ.get("ACC_DATA_ROOT", os.path.expanduser("~/acc_data"))


In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

In [ ]:
counts = pd.read_csv(f'{ACC_DATA_ROOT}/bulk_rna_data/counts_csv_file/bulk_counts.csv')
counts

In [ ]:
counts.columns

In [ ]:
counts = counts.rename(columns={'Unnamed: 0':'Geneid'})

In [ ]:
counts

In [ ]:
counts = counts.set_index('Geneid')
counts

In [ ]:
# genes with 0 count across all the columns will be removed. (60279 > 42660)
# counts = counts[counts.sum(axis = 1) > 0]
# counts

In [ ]:
# transposing the data because PyDESEq2 needs the sample information as rows.
counts = counts.T

In [ ]:
counts

In [ ]:
# metadata = pd.DataFrame(zip(counts.index, ['Avg', 'Avg', 'Avg', 'Avg', 'Avg', 'Avg', 'Avg', 'Avg', 'Avg', 'Avg', 'Poor', 'Avg', 'Avg', 'Poor', 'Poor', 'Poor', 'Avg', 'Avg', 'Poor', 'Poor']),
#                         columns = ['Sample', 'Condition'])

import pandas as pd

# Updated mapping based on clinical_info.csv (Good -> Good, Poor -> Poor)
conditions_mapped = [
    'Good', 'Good', 'Good', 'Good', 'Good', 'Good', 'Good', 'Good', 'Good', 'Good', 
    'Poor', 'Poor', 'Poor', 'Poor', 'Poor', 'Poor', 'Poor', 'Poor', 'Poor', 'Poor'
]

metadata = pd.DataFrame(
    zip(counts.index, conditions_mapped),
    columns=['Sample', 'Condition']
)

In [ ]:
metadata

In [ ]:
metadata = metadata.set_index('Sample')
metadata

In [ ]:
dds = DeseqDataSet(counts=counts,
            metadata=metadata,
            design_factors="Condition",
            n_cpus=1)

In [ ]:
dds.deseq2()

In [ ]:
dds

In [ ]:
dds.var

In [ ]:
# dds.write_h5ad(f'{ACC_DATA_ROOT}/bulk_rna_data/outputs/DEG.h5ad')

In [ ]:
stat_res = DeseqStats(dds, n_cpus=1, contrast = ('Condition','Good','Poor'))

In [ ]:
stat_res.summary()

In [ ]:
res = stat_res.results_df

In [ ]:
res

In [ ]:
from sanbomics.tools import id_map

In [ ]:
mapper = id_map(species = 'human')

In [ ]:
res['Symbol'] = res.index.map(mapper.mapper) 

In [ ]:
res

In [ ]:
res.to_csv(f'{ACC_DATA_ROOT}/bulk_rna_data/outputs/pydeseq2_out/clinical_outcome/DEG_results.csv')

In [ ]:
sigs = res[(res.padj < 0.05) & (abs(res.log2FoldChange) > 0.5)]

In [ ]:
sigs

In [ ]:
sigs.to_csv(f'{ACC_DATA_ROOT}/bulk_rna_data/outputs/pydeseq2_out/clinical_outcome/DEG_significant.csv')

In [ ]:
sigs